In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

In [4]:
df = pd.read_csv('D:\Logesh\Swiggy’s Restaurant Recommendation System using Streamlit\cleaned_swiggy (2).csv')
df

<>:1: SyntaxWarning: invalid escape sequence '\L'
<>:1: SyntaxWarning: invalid escape sequence '\L'
C:\Users\chuwi gemibook xpro\AppData\Local\Temp\ipykernel_10336\3677302968.py:1: SyntaxWarning: invalid escape sequence '\L'
  df = pd.read_csv('D:\Logesh\Swiggy’s Restaurant Recommendation System using Streamlit\cleaned_swiggy (2).csv')


,name,city,rating,rating_count,cost,cuisine,no_rating
0,AB FOODS POINT,Abohar,0.0,10,200.0,"Beverages,Pizzas",True
1,Janta Sweet House,Abohar,4.4,50,200.0,"Sweets,Bakery",False
2,theka coffee desi,Abohar,3.8,100,100.0,Beverages,False
3,Singh Hut,Abohar,3.7,20,250.0,"Fast Food,Indian",False
4,GRILL MASTERS,Abohar,0.0,10,250.0,"Italian-American,Fast Food",True
...,...,...,...,...,...,...,...
148440,The Food Delight,Yavatmal,0.0,10,200.0,"Fast Food,Snacks",True
148441,MAITRI FOODS & BEVERAGES,Yavatmal,0.0,10,300.0,Pizzas,True
148442,Cafe Bella Ciao,Yavatmal,0.0,10,300.0,"Fast Food,Snacks",True
148443,GRILL ZILLA,Yavatmal,0.0,10,250.0,Continental,True


In [5]:
# Create effective rating
df['effective_rating'] = df['rating'] * (df['rating_count'] / (df['rating_count'] + 50))

In [6]:
df.head()

,name,city,rating,rating_count,cost,cuisine,no_rating,effective_rating
0,AB FOODS POINT,Abohar,0.0,10,200.0,"Beverages,Pizzas",True,0.000000
1,Janta Sweet House,Abohar,4.4,50,200.0,"Sweets,Bakery",False,2.200000
2,theka coffee desi,Abohar,3.8,100,100.0,Beverages,False,2.533333
3,Singh Hut,Abohar,3.7,20,250.0,"Fast Food,Indian",False,1.057143
4,GRILL MASTERS,Abohar,0.0,10,250.0,"Italian-American,Fast Food",True,0.000000


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
# Keep display data separately
display_df = df[['name', 'city', 'cuisine', 'rating', 'cost']]

In [9]:
# Model data
model_df = df.drop(columns=['name'])

In [ ]:
# ======================================
# ENCODING (CITY + CUISINE)
# ======================================

In [22]:
categorical_cols = ['city', 'cuisine']

encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

encoded_cat = encoder.fit_transform(model_df[categorical_cols])

In [28]:
numerical_cols = ['effective_rating', 'rating_count', 'cost', 'no_rating']

In [29]:
from scipy.sparse import hstack

In [31]:

# Categorical
encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
encoded_cat = encoder.fit_transform(model_df[['city', 'cuisine']])

# Numerical
scaler = StandardScaler()
scaled_num = scaler.fit_transform(model_df[numerical_cols])

# Combine
final_df = hstack([encoded_cat, scaled_num])

In [35]:
final_df

<COOrdinate sparse matrix of dtype 'float64'
	with 890670 stored elements and shape (148445, 2958)>

In [32]:
kmeans = KMeans(n_clusters=6, random_state=42)
df['cluster'] = kmeans.fit_predict(final_df)

In [37]:
from scipy.sparse import csr_matrix

final_df = final_df.tocsr()

In [38]:
# Sample
sample_size = 10000
sample_indices = np.random.choice(final_df.shape[0], sample_size, replace=False)

# Now indexing works ✅
sample_data = final_df[sample_indices]
sample_labels = df['cluster'].iloc[sample_indices]

# Score
score = silhouette_score(sample_data, sample_labels)

print("Silhouette Score:", score)

Silhouette Score: 0.19529547235812816


In [45]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_within_cluster(index, top_n=5):
    
    # Get cluster of the selected restaurant
    cluster = df.loc[index, 'cluster']
    
    # Get indices of same cluster
    cluster_indices = df[df['cluster'] == cluster].index
    
    # Get query vector
    query = final_df[index]
    
    # Get cluster data
    cluster_data = final_df[cluster_indices]
    
    # Compute similarity
    scores = cosine_similarity(query, cluster_data).flatten()
    
    # Sort and get top indices
    top_indices = cluster_indices[scores.argsort()[::-1][1:top_n+1]]
    
    # Return recommendations
    return df.loc[top_indices][['name', 'city', 'cuisine', 'rating', 'cost']]

In [46]:
index = 100
print(df.iloc[index][['name', 'city', 'cuisine', 'rating', 'cost']])

name       NEW GANGOUR SWEETS (Adityapur)
city                            Adityapur
cuisine                      South Indian
rating                                4.0
cost                                200.0
Name: 100, dtype: object


In [50]:
recommend_within_cluster(299)

,name,city,cuisine,rating,cost
233,Apna Adda,Adityapur,"Chinese,Snacks",4.1,150.0
375,Anurag Fast Food Centre,Adityapur,"Chinese,Fast Food",4.1,150.0
428,Aman Momos,Adityapur,"Nepalese,Tibetan",4.2,150.0
275,Tinkoniya Hotel,Adityapur,"Snacks,Sweets",3.9,150.0
143,Kolkata Biryani - Jugsalai,Adityapur,Biryani,3.8,150.0


In [51]:
#Finalize Recommendation Function
def recommend(index, top_n=5):
    
    query = final_df[index]
    scores = cosine_similarity(query, final_df).flatten()
    
    top_indices = scores.argsort()[::-1][1:top_n+1]
    
    return df.iloc[top_indices][['name','city','cuisine','rating','cost']]

In [52]:
import pickle

pickle.dump(encoder, open("encoder.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(kmeans, open("kmeans.pkl", "wb"))

In [53]:
df.to_csv("final_data.csv", index=False)